# LangChain 기초

**학습 목표**

> 1. 이번 실습에서는 LLM 앱을 만들 때 가장 먼저 필요한 LangChain 기본 구성요소를 익힌다.
> 2. `init_chat_model`을 사용해 모델 식별자만 변경하여 OpenAI, Gemini, Ollama 간 전환을 수행한다.
> 3. **Messages → Prompt Template → Output Parser** 의 표준 처리 흐름을 익힌다.
> 4. **LLM을 호출하고, 프롬프트를 템플릿화하고, 결과를 원하는 형식으로 받는 법**을 익힌다.


---

> **LangChain v1.0의 변경점**
> - `langchain` 네임스페이스가 5개로 단순화됨: `langchain.messages`, `langchain.tools`, `langchain.agents`, `langchain.chat_models`, `langchain.embeddings`.
> - `LLMChain`, 전통 Retriever 등 레거시 기능은 `langchain-classic` 패키지로 분리됨.
> - 본 실습은 v1.0 이상의 API만 사용합니다.
---

# 1. 환경 준비


## (1) 라이브러리 설치

처음 실행하는 환경이라면 아래 셀의 주석을 해제하고 실행합니다. 이미 `pyproject.toml` 또는 `requirements.txt`로 설치했다면 실행하지 않아도 됩니다.

In [1]:
# 필요한 라이브러리 설치
%pip install -U langchain langchain-core langchain-openai langchain-google-genai langchain-groq langchain-ollama python-dotenv pydantic pandas

   ---------------------------------------- 0.0/551.7 kB ? eta -:--:--
   ---------------------------------------- 551.7/551.7 kB 10.7 MB/s  0:00:00

  Attempting uninstall: langchain-core

    Found existing installation: langchain-core 1.4.3

    Uninstalling langchain-core-1.4.3:

      Successfully uninstalled langchain-core-1.4.3

   ---------------------------------------- 0/2 [langchain-core]
   ---------------------------------------- 0/2 [langchain-core]
   ---------------------------------------- 0/2 [langchain-core]
   ---------------------------------------- 0/2 [langchain-core]
   ---------------------------------------- 0/2 [langchain-core]
   ---------------------------------------- 0/2 [langchain-core]
   ---------------------------------------- 0/2 [langchain-core]
   ---------------------------------------- 0/2 [langchain-core]
   ---------------------------------------- 0/2 [langchain-core]
   ---------------------------------------- 0/2 [langchain-core]
   ---------

## (2) 라이브러리 Import

이번 실습에서 사용하는 핵심 객체는 다음과 같습니다.

| 객체 | 역할 |
|---|---|
| `init_chat_model` | `provider:model` 문자열로 여러 제공자의 채팅 모델을 통합 초기화 |
| `PromptTemplate` | 문자열 기반 프롬프트 템플릿 |
| `ChatPromptTemplate` | system/human 메시지를 분리하는 채팅 프롬프트 |
| `StrOutputParser` | 모델 응답에서 문자열만 추출 |


In [2]:
import os
from pathlib import Path
from getpass import getpass
from typing import Literal

import pandas as pd
from dotenv import load_dotenv
from pydantic import BaseModel, Field

from langchain.chat_models import init_chat_model
from langchain_core.prompts import PromptTemplate, ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser

## (3) API Key 설정

API 키는 코드에 직접 작성하지 않습니다. 권장 방식은 `.env` 파일에 저장하는 것입니다.

```text
# OpenAI를 사용할 때
OPENAI_API_KEY=sk-...

# Gemini를 사용할 때
GOOGLE_API_KEY=...

```


In [3]:
load_dotenv()

print("OPENAI_API_KEY:", "있음" if os.getenv("OPENAI_API_KEY") else "없음")
print("GOOGLE_API_KEY:", "있음" if os.getenv("GOOGLE_API_KEY") else "없음")

OPENAI_API_KEY: 있음
GOOGLE_API_KEY: 있음


# 2. LangChain이 필요한 이유

OpenAI SDK나 다른 LLM API만으로도 모델 호출은 가능합니다. 예를 들어 "랭체인이 뭐야?"라고 바로 물어볼 수 있습니다.

하지만 실제 서비스나 업무 자동화에서는 단순 호출만으로 충분하지 않습니다.

| 필요 기능 | 실제 상황 예시 |
|---|---|
| 프롬프트 재사용 | 주제만 바꿔 같은 형식의 설명 생성 |
| 역할 부여 | 강사, 면접관, 상담사, 분석가 역할 지정 |
| 출력 형식 고정 | JSON, 표, 리스트, Pydantic 객체 |
| 여러 단계 연결 | 요약 -> 번역 -> 퀴즈 생성 |
| 대량 처리 | 고객 후기 100개를 같은 방식으로 분석 |
| 유지보수 | 프롬프트, 모델, 파서를 분리해 관리 |

LangChain은 LLM 호출을 **구조화된 실행 블록**으로 만들고, 이 블록들을 연결해 앱의 흐름을 구성하는 도구입니다.

# 3. Model

## (1) Model과 `init_chat_model`

- Model은 실제 답변을 생성하는 엔진
- LangChain에서는 모델을 공통 인터페이스로 감싸기 때문에, 이후 프롬프트나 파서와 쉽게 연결할 수 있음

- LangChain v1.x에서는 `init_chat_model("provider:model")` 한 함수로 여러 제공자의 채팅 모델을 통합 초기화할 수 있음.
    - 제공자별 import를 줄일 수 있습니다.
    - 모델 문자열만 바꾸면 OpenAI, Gemini, Groq, Ollama 등으로 전환할 수 있습니다.
    - 이후 `invoke`, `stream`, `batch`, Prompt 연결 방식은 동일하게 유지됩니다.


`CHAT_MODEL`은 `provider:model` 형식을 권장합니다.

| 제공자 | 예시 |
|---|---|
| OpenAI | `openai:gpt-4.1-mini` |
| Gemini | `google_genai:gemini-2.5-flash-lite` |
| Ollama | `ollama:gemma4:e4b` |

### [참고] GPT-5 시리즈는 temperature 고정

GPT-5 / 5.4 / 5.5 모델은 내부에서 multi-pass reasoning 을 수행하기 때문에 답의 무작위성을 막는 `temperature`·`top_p`·`logprob` 파라미터가 사실상 사용 불가 (temperature 는 1 로 고정). `init_chat_model("openai:gpt-5.4-mini")` 처럼 추가 인자 없이 부르면 됩니다.

답의 다양성·톤 조정은 **system 프롬프트**와 **few-shot 예시**로 합니다.

In [4]:
# CHAT_MODEL = 'openai:gpt-4.1-mini'
# CHAT_MODEL = 'google_genai:gemini-2.5-flash-lite'
CHAT_MODEL = 'openai:gpt-4.1-mini'
model = init_chat_model(CHAT_MODEL)
model

ChatOpenAI(output_version=None, profile={'name': 'GPT-4.1 mini', 'release_date': '2025-04-14', 'last_updated': '2025-04-14', 'open_weights': False, 'max_input_tokens': 1047576, 'max_output_tokens': 32768, 'text_inputs': True, 'image_inputs': True, 'audio_inputs': False, 'pdf_inputs': True, 'video_inputs': False, 'text_outputs': True, 'image_outputs': False, 'audio_outputs': False, 'video_outputs': False, 'reasoning_output': False, 'tool_calling': True, 'structured_output': True, 'attachment': True, 'temperature': True, 'image_url_inputs': True, 'pdf_tool_message': True, 'image_tool_message': True, 'tool_choice': True}, client=<openai.resources.chat.completions.completions.Completions object at 0x000001D2170D4590>, async_client=<openai.resources.chat.completions.completions.AsyncCompletions object at 0x000001D2170D5010>, root_client=<openai.OpenAI object at 0x000001D2166E3A10>, root_async_client=<openai.AsyncOpenAI object at 0x000001D2170D4D70>, model_name='gpt-4.1-mini', model_kwargs={

- `init_chat_model`로 만든 모델도 다른 채팅 모델과 동일하게 `invoke()`로 호출합니다. 반환값은 메시지 객체이며, 실제 답변 텍스트는 `.content`에서 확인합니다.

In [5]:
response = model.invoke('좀비가 카페를 운영한다면 첫날 메뉴판 5개를 짧게 뽑아줘.')
print(response.content)

물론이죠! 좀비가 운영하는 카페 첫날 메뉴판 5개, 간단히 뽑아봤어요:

1. **뇌맛 라떼**  
2. **피의 에스프레소**  
3. **썩은 과일 스무디**  
4. **모든 쓴맛의 죽음 차**  
5. **좀비 쿨러 (블러디 오렌지 주스)**  

이름만 봐도 좀비 감성이 느껴지네요!


## (2) `ChatOpenAI` 사용

In [6]:
from langchain_openai import ChatOpenAI

model = ChatOpenAI(
    model='gpt-4.1-mini',   # 사용할 OpenAI 모델 이름
    timeout=30,             # 응답 대기 시간 제한 : 30초
    max_retries=3,          # 요청 실패 시 최대 3번까지 재시도
)

model

ChatOpenAI(output_version=None, profile={'name': 'GPT-4.1 mini', 'release_date': '2025-04-14', 'last_updated': '2025-04-14', 'open_weights': False, 'max_input_tokens': 1047576, 'max_output_tokens': 32768, 'text_inputs': True, 'image_inputs': True, 'audio_inputs': False, 'pdf_inputs': True, 'video_inputs': False, 'text_outputs': True, 'image_outputs': False, 'audio_outputs': False, 'video_outputs': False, 'reasoning_output': False, 'tool_calling': True, 'structured_output': True, 'attachment': True, 'temperature': True, 'image_url_inputs': True, 'pdf_tool_message': True, 'image_tool_message': True, 'tool_choice': True}, client=<openai.resources.chat.completions.completions.Completions object at 0x000001D217188550>, async_client=<openai.resources.chat.completions.completions.AsyncCompletions object at 0x000001D217188F50>, root_client=<openai.OpenAI object at 0x000001D2171882D0>, root_async_client=<openai.AsyncOpenAI object at 0x000001D217188CD0>, model_name='gpt-4.1-mini', model_kwargs={

- `invoke()`는 하나의 입력을 넣고 하나의 출력을 받는 가장 기본적인 실행 방식입니다.

In [7]:
response = model.invoke("LangChain을 처음 배우는 사람에게 아주 친절하게 한번 봐도 이해가 될 정도로 한 문단으로 설명해줘")
print(response.content)

LangChain은 여러 다양한 데이터나 도구들과 쉽게 연결해서, 인공지능 모델이 더 똑똑하게 정보를 이해하고 처리하도록 도와주는 소프트웨어 프레임워크입니다. 예를 들어, 인터넷에서 문서를 가져오거나, 데이터베이스에 있는 정보를 읽어오고, 그 정보를 바탕으로 AI가 질문에 답하거나 요약을 할 수 있게 해주는데, 복잡한 코딩 없이도 이런 작업을 쉽고 체계적으로 할 수 있게 만들어줍니다. 그래서 AI를 활용한 챗봇이나 문서 분석 같은 프로젝트를 처음 시작하는 사람도 단계별로 따라 하면서 빠르게 결과를 내볼 수 있답니다!


- 응답 객체에는 답변 본문뿐 아니라 모델명, 토큰 사용량 같은 메타데이터가 포함될 수 있습니다. 
- 실제 서비스에서는 사용량 추적이나 로깅에 활용할 수 있습니다.

In [8]:
print("content:", response.content)
print("response_metadata:", response.response_metadata)
print("usage_metadata:", response.usage_metadata)

content: LangChain은 여러 다양한 데이터나 도구들과 쉽게 연결해서, 인공지능 모델이 더 똑똑하게 정보를 이해하고 처리하도록 도와주는 소프트웨어 프레임워크입니다. 예를 들어, 인터넷에서 문서를 가져오거나, 데이터베이스에 있는 정보를 읽어오고, 그 정보를 바탕으로 AI가 질문에 답하거나 요약을 할 수 있게 해주는데, 복잡한 코딩 없이도 이런 작업을 쉽고 체계적으로 할 수 있게 만들어줍니다. 그래서 AI를 활용한 챗봇이나 문서 분석 같은 프로젝트를 처음 시작하는 사람도 단계별로 따라 하면서 빠르게 결과를 내볼 수 있답니다!
response_metadata: {'token_usage': {'completion_tokens': 147, 'prompt_tokens': 35, 'total_tokens': 182, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-4.1-mini-2025-04-14', 'system_fingerprint': 'fp_4f7ecf0cc2', 'id': 'chatcmpl-DpNpXdaaQ3wtf8oicBIcjUNvzuUND', 'service_tier': 'default', 'finish_reason': 'stop', 'logprobs': None}
usage_metadata: {'input_tokens': 35, 'output_tokens': 147, 'total_tokens': 182, 'input_token_details': {'audio': 0, 'cache_read': 0}, 'output_token_details': {'aud

## (3) Temperature 비교

- `temperature`는 답변의 다양성과 예측 가능성을 조절하는 값

| 값 | 특징 |
|---|---|
| 0에 가까움 | 안정적, 반복 실행 시 비슷한 결과 |
| 1에 가까움 | 다양하고 창의적인 결과 |


In [9]:
question = '생성형 ai를 중학생에게 설명해줘.'

cold_model = init_chat_model(CHAT_MODEL, temperature=0)
creative_model = init_chat_model(CHAT_MODEL, temperature=1.0)

In [10]:

print("[temperature=0]")
print(cold_model.invoke(question).content)

print("\n[temperature=1]")
print(creative_model.invoke(question).content)

[temperature=0]
물론이지! 중학생 친구에게 쉽게 설명해볼게.

생성형 AI는 '새로운 것들을 만들어 내는 인공지능'이야. 예를 들어, 그림을 그리거나, 글을 쓰거나, 음악을 만드는 것처럼 사람이 하는 창작 활동을 컴퓨터가 할 수 있게 만든 거야.

이 AI는 많은 데이터를 배우고, 그걸 바탕으로 새로운 내용을 만들어 내는 능력이 있어. 예를 들어, 수천 권의 책을 읽고 나서 새로운 이야기를 쓰거나, 수많은 그림을 보고 나서 새로운 그림을 그릴 수 있는 거지.

쉽게 말하면, 생성형 AI는 '컴퓨터가 스스로 상상해서 새로운 것들을 만들어 내는 똑똑한 친구'라고 생각하면 돼!

[temperature=1]
물론이야! 중학생 친구가 이해하기 쉽게 생성형 AI에 대해 설명해 줄게.

---

**생성형 AI란?**

생성형 AI는 컴퓨터가 사람처럼 새로운 글, 그림, 음악, 이야기 등을 만들어 내는 똑똑한 프로그램이에요.

예를 들어, 누군가가 "고양이 그림 그려줘"라고 말하면, 생성형 AI는 그 말에 맞는 고양이 그림을 새롭게 만들어 줄 수 있어요. 또, "재미있는 이야기를 써줘" 하면 이야기도 직접 만들어 낼 수 있답니다.

---

**어떻게 작동할까?**

생성형 AI는 아주 많은 데이터(글, 사진, 음악 등)를 공부해요. 그러고 나서 그 경험을 바탕으로 새로운 것을 창조하는 거예요. 마치 우리가 책을 많이 읽고, 그림을 많이 그려서 더 잘 만드는 것처럼요.

---

**왜 중요한가?**

- 창의적인 일을 도와줘요. 예를 들어, 학생들이 글쓰기 숙제를 할 때 아이디어를 줄 수 있어요.
- 사람들이 더 쉽게 무언가를 만들 수 있게 도와줘요.
- 앞으로 더 많은 분야에서 사람들이 일을 편리하게 할 수 있도록 도와줄 거예요.

---

이해하기 쉬웠으면 좋겠다! 궁금한 게 있으면 언제든 물어봐.


## [실습] 모델 호출 바꿔보기

모델을 gemini 등 다른 모델로 바꿔봅니다.
그리고 아래 질문을 바꿔 실행해 봅니다.

- "RAG를 비전공자에게 설명해줘."
- "AI Agent를 회사 업무 자동화 예시로 설명해줘."
- "LangGraph와 LangChain의 차이를 간단히 설명해줘."

In [11]:
question2 = 'RAG를 비전공자에게 설명해줘.'


print("[temperature=0]")
print(cold_model.invoke(question2).content)

print("\n[temperature=1]")
print(creative_model.invoke(question2).content)

[temperature=0]
물론이죠! RAG에 대해 비전공자도 이해하기 쉽게 설명해드릴게요.

---

### RAG란 무엇인가요?

RAG는 **"Retrieval-Augmented Generation"**의 약자입니다. 쉽게 말해, **필요한 정보를 찾아서 그 정보를 바탕으로 답을 만들어내는 인공지능 기술**입니다.

---

### 왜 필요할까요?

우리가 궁금한 게 있을 때, 인터넷에서 정보를 찾아보고 그걸 바탕으로 답을 만들잖아요? 인공지능도 마찬가지예요. 하지만 인공지능이 혼자서 모든 정보를 다 알고 있지는 않아요. 그래서 외부에서 필요한 정보를 찾아와서 더 정확하고 풍부한 답을 만들도록 도와주는 방법이 바로 RAG입니다.

---

### 어떻게 작동하나요?

1. **질문을 받으면**  
   인공지능이 먼저 관련된 정보를 외부 데이터베이스나 문서에서 찾아요.  
   (예: 책, 논문, 웹사이트 등)

2. **찾은 정보를 바탕으로**  
   그 정보를 참고해서 답변을 만들어내요.

---

### 비유로 설명하면?

- **도서관 사서와 대화하는 것과 비슷해요.**  
  당신이 어떤 질문을 하면, 사서가 도서관에서 관련 책을 찾아서 내용을 알려주고, 그걸 바탕으로 당신에게 답을 해주는 거죠.

---

### 정리

- RAG는 인공지능이 **필요한 정보를 찾아서**  
- 그 정보를 이용해 **더 정확하고 풍부한 답변을 만드는 기술**입니다.

---

궁금한 점 있으면 언제든 물어보세요!

[temperature=1]
물론입니다! RAG는 **"Retrieval-Augmented Generation"**의 약자입니다. 쉽게 말해, 컴퓨터가 더 똑똑하게 답을 찾도록 돕는 기술이에요.

조금 더 풀어서 설명하자면:

- **Retrieval(검색)**: 컴퓨터가 어떤 질문을 받았을 때, 미리 저장된 큰 자료(문서, 책, 웹페이지 등)에서 관련된 정보를 찾아옵니다.
- **Augmented Generation(생성 보강)**: 그 찾아온 

# 4. Prompt

- Prompt는 LLM에게 주는 작업 지시서
- 좋은 프롬프트는 단순히 질문을 잘 쓰는 것이 아니라, 다음 요소를 설계하는 일

| 요소 | 예시 |
|---|---|
| 역할 | "너는 AI 강의를 하는 친절한 강사야." |
| 목적 | "초보자가 이해할 수 있게 설명해줘." |
| 제약 | "5문장 이내로 작성해줘." |
| 출력 형식 | "표로 정리해줘.", "JSON으로 답해줘." |
| 예시 | "아래 예시와 같은 톤으로 작성해줘." |

LangChain에서는 반복되는 프롬프트를 템플릿으로 만들고, 변수만 바꿔 재사용합니다.

## (1) PromptTemplate

- `PromptTemplate`은 문자열 기반 템플릿
- `{topic}` 같은 변수를 넣고, 실행할 때 실제 값을 전달

In [12]:
basic_prompt = PromptTemplate.from_template(
    '''
다음 개념을 초등학생도 이해할 수 있게 설명해줘.

개념 : {topic}

조건 :
- 어려운 용어는 풀어서설명한다.
- 일상생활 비유를 1개 포함한다.
- 마지막에 한 줄 요약을 작성한다.
'''.strip()
)

formatted_prompt = basic_prompt.invoke({"topic":"Self-Attention"})

# 아직 LLM 호출한 것은 아니고
print(formatted_prompt.text)

다음 개념을 초등학생도 이해할 수 있게 설명해줘.

개념 : Self-Attention

조건 :
- 어려운 용어는 풀어서설명한다.
- 일상생활 비유를 1개 포함한다.
- 마지막에 한 줄 요약을 작성한다.


## (2) ChatPromptTemplate

- 채팅 모델에는 `ChatPromptTemplate`이 더 자주 사용됨
- 시스템 메시지, 사용자 메시지, AI 메시지 등 역할(role) 구분
- 다중 메시지 기반의 프롬프트 흐름을 구성할 수 있도록 도와주는 템플릿

| 메시지 역할 | 의미 |
|---|---|
| `System` | 모델의 역할, 원칙, 톤. AI에게 역할/성격을 지정 |
| `Human` | 실제 사용자 질문 또는 요청 |
| `AI` | AI 응답 |

In [13]:
chat_prompt = ChatPromptTemplate.from_messages([
('system', '너는 생성형 AI와 Langchain에 대해 쉽게 설명을 해주는 멘토다. 한국어로 대답해줘'),
('human', '주제: {topic}\n 대상: {audience}\n 요청: 쉬운 설명, 예시 2개, 확인 질문 1개를 작성해줘.')
])

messages = chat_prompt.invoke({
    'topic':'프롬프트 엔지니어링',
    'audience':'LLM을 처음 배우는 비전공자'
})

messages

ChatPromptValue(messages=[SystemMessage(content='너는 생성형 AI와 Langchain에 대해 쉽게 설명을 해주는 멘토다. 한국어로 대답해줘', additional_kwargs={}, response_metadata={}), HumanMessage(content='주제: 프롬프트 엔지니어링\n 대상: LLM을 처음 배우는 비전공자\n 요청: 쉬운 설명, 예시 2개, 확인 질문 1개를 작성해줘.', additional_kwargs={}, response_metadata={})])

In [14]:
result = model.invoke(messages)
print(result.content)

안녕하세요! 오늘은 '프롬프트 엔지니어링'에 대해 쉽게 설명해 드릴게요.

---

### 프롬프트 엔지니어링이란?

프롬프트 엔지니어링은 **AI에게 원하는 답변을 정확하게 얻기 위해 질문이나 명령어(프롬프트)를 잘 만드는 방법**이에요. 즉, AI가 이해하기 쉽게, 그리고 원하는 정보를 잘 뽑아낼 수 있도록 문장을 만드는 기술이죠.

---

### 왜 중요할까요?

AI는 우리가 입력한 프롬프트에 따라 결과가 크게 달라져요. 좋은 프롬프트를 작성하면 AI에게 더 정확하고 도움이 되는 답변을 받을 수 있답니다.

---

### 예시 1: 단순 프롬프트 vs. 개선된 프롬프트

- 단순 프롬프트:  
  "서울 날씨 알려줘."

- 개선된 프롬프트:  
  "오늘 서울의 오전 9시부터 오후 6시까지의 날씨 예보를 상세히 알려줘."

**결과 차이:**  
단순 프롬프트는 간단히 날씨만 알려주지만, 개선된 프롬프트는 구체적인 시간대와 상세한 정보를 받을 수 있어요.

---

### 예시 2: 역할과 조건을 포함한 프롬프트

- 단순 프롬프트:  
  "영어 이메일 써줘."

- 개선된 프롬프트:  
  "친구에게 여행 초대 메일을 영어로, 친근하고 밝은 톤으로 150자 내외로 작성해줘."

**결과 차이:**  
역할과 스타일, 글자 수 조건까지 넣으니 더 원하는 스타일의 이메일을 받을 수 있어요.

---

### 확인 질문

"프롬프트 엔지니어링이 왜 중요한지, 그리고 좋은 프롬프트를 만들 때 어떤 점을 신경 써야 하는지 한 문장으로 설명해 볼 수 있나요?"

---

필요하면 더 자세히 알려드릴게요!


## [실습] 역할 프롬프트 만들기

`system` 역할을 바꿔 같은 주제의 답변이 어떻게 달라지는지 확인합니다.

예시 역할:

- "너는 초등학생에게 설명하는 과학 선생님이다."
- "너는 기업 임원에게 보고하는 AI 컨설턴트다."
- "너는 개발자에게 코드 중심으로 설명하는 시니어 엔지니어다."

In [15]:
chat_prompt = ChatPromptTemplate.from_messages([
('system', '너는 초등학생에게 설명하는 과학 선생님이다. 한국어로 대답해줘'),
('human', '주제: {topic}\n 대상: {audience}\n 요청: 쉬운 설명, 예시 2개, 확인 질문 1개를 작성해줘.')
])

message2 = chat_prompt.invoke({
    'topic':'프롬프트 엔지니어링',
    'audience':'LLM을 처음 배우는 초등학생'
})

result = model.invoke(message2)
print(result.content)

안녕 친구들! 오늘은 ‘프롬프트 엔지니어링’에 대해 아주 쉽게 알아볼 거예요.

**프롬프트 엔지니어링이란?**  
컴퓨터한테 질문하거나 시킬 말을 똑똑하게 알려주는 방법이에요. 컴퓨터가 우리가 원하는 대답을 잘 할 수 있게 도와주는 거예요.

**쉬운 예시 2가지:**  
1. 친구에게 “안녕!”하고 인사할 때, 그냥 “안녕”이라고 말하면 친구가 웃으며 “안녕!” 하고 대답하잖아요. 그런데 “안녕, 오늘 기분 어때?”라고 말하면 친구가 기분도 알려줄 거예요. 이렇게 컴퓨터에게도 더 자세히 말해 주는 게 프롬프트 엔지니어링이에요.  
2. 숙제 도와달라고 할 때 “나무에 대해 알려줘” 대신에 “나무가 왜 초록색 잎을 가지고 있는지 알려줘”라고 물으면 컴퓨터가 더 정확한 답을 줘요.

**확인 질문:**  
컴퓨터에게 더 좋은 대답을 듣고 싶으면, 우리가 하는 말을 어떻게 해야 할까요? 

답을 생각해보고 알려줘요!


## [실습] 영화 추천 템플릿 만들기
- 입력변수 : 장르
- 장르를 입력받아 영화 1편과 추천이유를 설명하는 템플릿을 만들고 사용해 봅시다.

In [16]:
chat_prompt = ChatPromptTemplate.from_messages([
('system', '너는 시청자에게 설명하는 영화 평론가이다. 한국어로 대답해줘'),
('human', '주제: {topic}\n 대상: {audience}\n 장르: {genre}\n 요청: 쉬운 설명, 추천 영화 1개, 확인 질문 1개를 작성해줘.')
])

message3 = chat_prompt.invoke({
    'topic':'영화',
    'audience':'영화를 좋아하는 남자 중학생',
    'genre':'누아르'
})

result = model.invoke(message3)
print(result.content)

안녕! 누아르 영화는 주로 어두운 분위기와 복잡한 이야기를 가진 영화야. 주인공은 보통 나쁜 상황에 빠져서 문제를 해결하려고 노력하지만, 항상 쉽지 않아. 그림자와 빛의 대비가 뚜렷해서 화면이 멋지고 긴장감도 많아.

누아르 영화 중에서 '칠드런 오브 맨(Children of Men)'을 추천해줄게. 미래가 암울하고 주인공이 힘든 상황에서 싸우는 이야기를 그리고 있어.

궁금한 게 있어. 너는 어둡고 복잡한 이야기를 좋아해? 아니면 밝고 재미있는 이야기를 더 좋아해?


# 5. Output Parser

- Output Parser는 LLM에서 반환된 자유형 텍스트(string)를 우리가 원하는 형태로 가공해주는 도구

- LLM의 응답은 기본적으로 메시지 객체이고, 사람이 읽을 때는 `response.content`만 확인하면 되지만, 프로그램에서는 결과를 일정한 형태로 다루는 것이 중요함


| Parser | 역할 | 사용 상황 |
|---|---|---|
| `StrOutputParser` | 응답에서 문자열만 추출 | 일반 답변, 이메일, 요약 |
| JSON 계열 Parser | JSON 문자열을 dict로 변환 | API 응답처럼 쓰고 싶을 때 |
| Pydantic 구조화 출력 | 스키마에 맞는 객체로 검증 | 분류, 추출, 업무 자동화 |

최근 LangChain에서는 모델의 `with_structured_output()` 기능을 사용해 Pydantic 모델로 결과를 받는 방식도 많이 사용합니다.

## (1) StrOutputParser

`StrOutputParser`를 체인 끝에 붙이면 모델 응답 객체에서 텍스트만 꺼내 줍니다.

In [17]:
chat_prompt.input_variables

['audience', 'genre', 'topic']

In [21]:
# 1. 프롬프트 템플릿에 값을 넣어 실제 메시지 작성
messages = chat_prompt.invoke({
    'audience': '파이썬 기초를 아는 학생',
    'genre': '컴퓨터 교육',
    'topic': 'Output Parser'
})

print(messages.to_string())

# 2. 모델 호출
response = model.invoke(messages)

# 3. 모델 응답에서 문자열만 추출
parser = StrOutputParser()
answer = parser.invoke(response)

print(answer)

System: 너는 시청자에게 설명하는 영화 평론가이다. 한국어로 대답해줘
Human: 주제: Output Parser
 대상: 파이썬 기초를 아는 학생
 장르: 컴퓨터 교육
 요청: 쉬운 설명, 추천 영화 1개, 확인 질문 1개를 작성해줘.
안녕하세요! 오늘은 ‘Output Parser’에 대해 쉽게 설명해 드릴게요.

Output Parser란 프로그램이 만들어내는 결과물을 우리가 원하는 형태로 바꾸는 도구입니다. 예를 들어, 파이썬 프로그램이 어떤 복잡한 데이터를 출력할 때, 그 데이터를 보기 쉽고 다루기 좋은 형식으로 정리해 주는 역할을 합니다. 마치 영어 문장을 우리말로 깔끔하게 번역해 주는 것과 비슷하죠.

파이썬에서 Output Parser는 문자열을 잘게 쪼개서 필요한 정보만 추출하거나, JSON과 같은 데이터 형식을 파이썬 객체로 바꿔주는 기능을 합니다. 이렇게 하면 결과물을 프로그램 안에서 쉽게 사용할 수 있어요.

추천 영화로는 『매트릭스』(The Matrix, 1999)를 추천해요. 영화 속에서 현실과 가상 세계의 경계가 모호해지는 것처럼, 프로그램에서 ‘출력’을 원하는 형태로 변환하는 Output Parser의 역할도 비슷한 맥락에서 이해할 수 있답니다.

마지막으로 확인 질문!  
① Output Parser가 왜 필요한가요?  
② 파이썬에서 Output Parser를 이용하면 어떤 점이 편리한가요?

이 질문에 답해 보면서 이해도를 한 번 점검해 보세요!


In [24]:
parser = StrOutputParser()

simple_chain = chat_prompt | model | parser

answer = simple_chain.invoke({
    'audience': '파이썬 기초를 아는 학생',
    'genre': '컴퓨터 교육',
    'topic': 'Output Parser'
})

answer

"안녕하세요! 오늘은 'Output Parser'에 대해 쉽게 설명해드릴게요.\n\nOutput Parser란, 프로그램이나 함수가 만들어낸 결과물(출력값)을 사람이 읽기 좋게 변환하거나, 다른 프로그램이 이해할 수 있도록 정리해주는 도구를 말해요. 예를 들어, 파이썬에서 어떤 함수가 숫자나 문자열을 출력했을 때, 그 출력값을 정리해서 원하는 형태로 바꾸는 과정을 생각하면 돼요.\n\n쉽게 말해, Output Parser는 '출력물을 해석하고 필요한 정보만 뽑아내는 역할'이라고 볼 수 있어요. 그래서 데이터를 다루거나, 자동화 프로그램을 만들 때 매우 유용합니다.\n\n추천 영화: 「매트릭스」 (The Matrix, 1999)  \n이 영화는 현실과 컴퓨터 코드 사이의 경계를 탐구하는데, 우리가 데이터를 해석하고 처리하는 Output Parser의 역할과 비슷하게 ‘정보를 해석하는 과정’에 대해 생각할 수 있어요.\n\n확인 질문:  \nOutput Parser가 하는 역할은 무엇인가요?  \n\n답변해 보시고 궁금한 점 있으면 언제든 질문해 주세요!"

## (2) PydanticOutputParser

In [19]:
from pydantic import BaseModel
from langchain_core.output_parsers import StrOutputParser, JsonOutputParser, PydanticOutputParser

#### 1) Pydantic
- 파이썬에서 데이터 형태를 정의하고 검증하는 라이브러리

In [25]:
# 사용자 정보를 담는 Pydantic 모델을 정의
class User(BaseModel):
    name: str       # 사용자의 이름을 문자열(str)로 지정
    age: int        # 사용자의 나이를 정수(int)로 지정

#### 2) 출력파서로 이용
- llm의 성능에 따라 출력 파싱에 맞게 적절한  답변을 할 수도 있고, 잘못 답변해서 오류가 날 수도 있음.

In [ ]:
# 1. Pydantic 모델 정의
# 이상한 값이 들어오면 에러를 내거나, 가능한 경우 타입을 맞춰줌
class BookInfo(BaseModel):      # BaseModel을 상속하면 데이터 구조와 타입 검증
    title: str
    author: str
    year: int

# 2. 파서 생성
parser = PydanticOutputParser(pydantic_object=BookInfo)

# 3. 프롬프트 구성
prompt = ChatPromptTemplate([
    ('system', '너는 책 추천 전문가야'),
    ('human', '좋은 책 하나만 추천해줘. 제목과 저자, 출판년도 알려줘'),
    ('system', '{output_format}')
])

# 4. 메시지 생성
messages = prompt.format_messages(
    output_format = parser.get_format_instructions()
)

# 5. LLM 호출 및 파성
response = model.invoke(messages)
book = parser.parse(response.content)

# 6. 결과 출력
print(book)

title='데미안' author='헤르만 헤세' year=1919


In [28]:
messages

[SystemMessage(content='너는 책 추천 전문가야', additional_kwargs={}, response_metadata={}),
 HumanMessage(content='좋은 책 하나만 추천해줘. 제목과 저자, 출판년도 알려줘', additional_kwargs={}, response_metadata={}),
 SystemMessage(content='The output should be formatted as a JSON instance that conforms to the JSON schema below.\n\nAs an example, for the schema {"properties": {"foo": {"title": "Foo", "description": "a list of strings", "type": "array", "items": {"type": "string"}}}, "required": ["foo"]}\nthe object {"foo": ["bar", "baz"]} is a well-formatted instance of the schema. The object {"properties": {"foo": ["bar", "baz"]}} is not well-formatted.\n\nHere is the output schema:\n```\n{"properties": {"title": {"title": "Title", "type": "string"}, "author": {"title": "Author", "type": "string"}, "year": {"title": "Year", "type": "integer"}}, "required": ["title", "author", "year"]}\n```', additional_kwargs={}, response_metadata={})]

## (3) 구조화 출력

업무 자동화에서는 자유 텍스트보다 정해진 구조가 더 유용할 때가 많습니다.

예를 들어 고객 후기를 분석한다면 다음처럼 감성, 카테고리, 우선순위, 다음 조치를 분리해서 받아야 이후 시스템에서 활용하기 쉽습니다.

In [30]:
class CustomerIssue(BaseModel):
    sentiment: Literal['positive', 'neutral', 'negative'] = Field(
        description='고객 후기의 전체 감성')
    category: Literal['배송', '결제', '상품', 'CS', '기타'] = Field(
        description='가장 중요한 이슈 카테고리')
    priority: int = Field(
        description='1은 낮음, 5는 매우 긴급',
        ge = 1,
        le = 5
    )
    summary: str = Field(description='고객 이슈 요약')
    next_action: str = Field(description='담당자가 취해야 할 다음 조치')

# with_structured_output : LLM 출력이 Pydantic 모델 형태로 반환
issue_model = model.with_structured_output(CustomerIssue, method='json_schema')

review = '배송은 빨랐지만 박스가 찢어져 있고, 문의를 했지만 고객센터 답변이 3일째 없습니다.'
issue = issue_model.invoke(review)

issue

CustomerIssue(sentiment='negative', category='CS', priority=1, summary='배송 박스가 찢어져 도착했고, 고객센터 문의에도 3일째 답변이 지연되고 있음', next_action='고객센터에 빠른 답변 요청 및 피해 보상 안내')

In [31]:
issue.model_dump()

{'sentiment': 'negative',
 'category': 'CS',
 'priority': 1,
 'summary': '배송 박스가 찢어져 도착했고, 고객센터 문의에도 3일째 답변이 지연되고 있음',
 'next_action': '고객센터에 빠른 답변 요청 및 피해 보상 안내'}

## [실습] 감성 분석 결과 구조화

아래 리뷰를 분석해 다음 필드를 가진 `MovieReview` 모델을 만들어 봅니다.

입력 문장:

```text
이 영화는 영상미는 좋았지만 스토리가 너무 지루했다.
```

출력 목표:

```json
{
  "sentiment": "mixed",
  "positive": "영상미가 좋음",
  "negative": "스토리가 지루함",
  "recommendation": "시각적 연출을 좋아하는 관객에게만 추천"
}
```

In [34]:
class MovieReview(BaseModel):
    sentiment: Literal['positive', 'mixed', 'negative'] = Field(
        description='고객 후기의 전체 감성')
    Positrive: str = Field(
        description='긍정적인 의미만 정리')
    Negative: str = Field(
        description='부정적인 의미만 정리')
    Recommendation: str = Field(description='어떤 고객에게 추천하면 좋을지 요약')

# with_structured_output : LLM 출력이 Pydantic 모델 형태로 반환
issue_model = model.with_structured_output(MovieReview, method='json_schema')

review = '이 영화는 영상미는 좋았지만 스토리가 너무 지루했다.'
issue = issue_model.invoke(review)

issue

MovieReview(sentiment='mixed', Positrive='영상미가 뛰어나서 시각적으로 매우 만족스러웠다.', Negative='스토리가 지루해서 몰입하기 어려웠다.', Recommendation='스토리에 더 흥미를 느끼는 관객에게는 추천하지 않는다.')

In [35]:
issue.model_dump()

{'sentiment': 'mixed',
 'Positrive': '영상미가 뛰어나서 시각적으로 매우 만족스러웠다.',
 'Negative': '스토리가 지루해서 몰입하기 어려웠다.',
 'Recommendation': '스토리에 더 흥미를 느끼는 관객에게는 추천하지 않는다.'}

## [실습] 게임 캐릭터 카드
- 게임 캐릭터 이름, 직업, 성격, 대표 특기 또는 필살기, 약점을 나타내는 게임 캐릭터 카드를 만들어보세요.

In [38]:
class CharacterCard(BaseModel):
    Name: str = Field(
        description='어울리는 게임 캐릭터 이름')
    Job: str = Field(
        description='어울리는 게임 직업')
    Personality: str = Field(
        description='어울리는 캐릭터 성격')
    Special_Move: str = Field(
        description='이 캐릭터가 가장 잘하는 것')
    Weakness: str = Field(description='이 캐릭터가 가장 약한 부분')

# with_structured_output : LLM 출력이 Pydantic 모델 형태로 반환
issue_model = model.with_structured_output(CharacterCard, method='json_schema')

review = '이 캐릭터는 메이플스토리에서 성역에서 전투원을 힐링해주는 캐릭터다.'
issue = issue_model.invoke(review)

issue

CharacterCard(Name='엔젤릭버스터', Job='힐러', Personality='따뜻하고 배려심이 깊으며 동료들을 위해 헌신적인 성격', Special_Move='성스러운 빛의 파동으로 아군 전체를 치유하고 보호막을 부여함', Weakness='물리 공격에는 강하지만 마법 공격에 상대적으로 약함')

In [39]:
issue.model_dump()

{'Name': '엔젤릭버스터',
 'Job': '힐러',
 'Personality': '따뜻하고 배려심이 깊으며 동료들을 위해 헌신적인 성격',
 'Special_Move': '성스러운 빛의 파동으로 아군 전체를 치유하고 보호막을 부여함',
 'Weakness': '물리 공격에는 강하지만 마법 공격에 상대적으로 약함'}

## [실습] 장르 통한 영화 카드
- 영화 제목, 감독, 주연, 배우, 개봉년도을 나타내는 영화 카드를 만들어보세요.

In [42]:
# 1. Pydantic 모델 정의
# 이상한 값이 들어오면 에러를 내거나, 가능한 경우 타입을 맞춰줌
class MovieInfo(BaseModel):      # BaseModel을 상속하면 데이터 구조와 타입 검증
    title: str
    producer: str
    Main_actor: str
    actors: str
    year: int

# 2. 파서 생성
parser = PydanticOutputParser(pydantic_object=MovieInfo)

# 3. 프롬프트 구성
prompt = ChatPromptTemplate([
    ('system', '너는 장르에 따라 영화를 추천해주는 장르별 영화 추천 전문가야'),
    ('human', '누아르 영화를 추천 받고 싶어. 영화 제목과, 감독, 주연, 배우, 개봉년도 알려줘'),
    ('system', '{output_format}')
])

# 4. 메시지 생성
messages = prompt.format_messages(
    output_format = parser.get_format_instructions()
)

# 5. LLM 호출 및 파성
response = model.invoke(messages)
movie = parser.parse(response.content)

# 6. 결과 출력
print(movie)

title='Chinatown' producer='Roman Polanski' Main_actor='Jack Nicholson' actors='Faye Dunaway, John Huston' year=1974
